In [1]:
import pyodbc
import pandas as pd
import json
from tqdm import tqdm

In [2]:
# Connexions
conn_stg = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=Staging_Test;"
    "UID=moeness;"
    "PWD=azerty"
)
conn_dwh = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=DWH_ClubAfricain;"
    "UID=moeness;"
    "PWD=azerty"
)

In [3]:
def upsert_dimension_from_config(config, conn_stg, conn_dwh):
    table_name = config["table"]
    source_table = config["source"]
    columns = config["columns"]
    is_static = config.get("static", False)

    cursor_dwh = conn_dwh.cursor()

    # Vérifie si la dimension statique existe déjà
    if is_static:
        check_sql = f"""
        SELECT COUNT(*) FROM INFORMATION_SCHEMA.TABLES
        WHERE TABLE_NAME = '{table_name}'
        """
        cursor_dwh.execute(check_sql)
        exists = cursor_dwh.fetchone()[0]
        if exists:
            print(f"✅ Dimension statique '{table_name}' existe déjà. Ignorée.")
            return

    # Lecture dynamique avec ou sans colonne 'Actif'
    try:
        test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
        if "Actif" in test_df.columns:
            df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
        else:
            df = pd.read_sql(f"SELECT * FROM {source_table}", conn_stg)
    except Exception as e:
        print(f"❌ Erreur lecture source {source_table} : {e}")
        return

    df = df[columns]

    # Création de la table dans le DWH si elle n'existe pas
    dtype_map = {
        "int64": "INT", "float64": "FLOAT", "object": "VARCHAR(255)", "bool": "BIT"
    }

    create_cols = []
    pk_col = columns[0]  # On suppose que la première colonne est la PK

    for col in df.columns:
        dtype = dtype_map.get(str(df[col].dtype), "VARCHAR(255)")
        line = f"{col} {dtype}"
        if col == pk_col:
            line += " PRIMARY KEY"
        create_cols.append(line)

    create_sql = f"""
    IF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = '{table_name}')
    BEGIN
        CREATE TABLE {table_name} (
            {', '.join(create_cols)}
        )
    END
    """
    cursor_dwh.execute(create_sql)
    conn_dwh.commit()

    # UPSERT (update si existe, insert sinon)
    for _, row in df.iterrows():
        values = [None if pd.isna(v) else v for v in row]
        pk_value = values[0]
        columns_sql = ", ".join(df.columns)
        placeholders = ", ".join(["?"] * len(values))
        updates_sql = ", ".join([f"{col} = ?" for col in df.columns[1:]])

        check_sql = f"SELECT COUNT(*) FROM {table_name} WHERE {pk_col} = ?"
        cursor_dwh.execute(check_sql, pk_value)
        exists = cursor_dwh.fetchone()[0]

        if exists:
            update_sql = f"UPDATE {table_name} SET {updates_sql} WHERE {pk_col} = ?"
            update_values = values[1:] + [pk_value]
            cursor_dwh.execute(update_sql, *update_values)
        else:
            insert_sql = f"INSERT INTO {table_name} ({columns_sql}) VALUES ({placeholders})"
            cursor_dwh.execute(insert_sql, *values)

    conn_dwh.commit()
    print(f"✅ Dimension '{table_name}' synchronisée avec mise à jour intelligente.")


In [4]:
def clean_sql_value(v):
    if pd.isna(v) or str(v).strip() == "":
        return None
    try:
        return float(v) if isinstance(v, float) or str(v).replace('.', '', 1).isdigit() else v
    except:
        return v

In [5]:
def upsert_fact_from_config(config, conn_stg, conn_dwh):
    import pandas as pd

    table_name = config["table"]
    source_table = config["source"]
    columns = config["columns"]
    join_keys = config.get("join_keys", {})
    auto_pk = config.get("auto_pk", False)
    pk_name = config.get("pk", f"ID_{table_name}")

    cursor_dwh = conn_dwh.cursor()

    # 1. Lire les données source
    try:
        test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
        if "Actif" in test_df.columns:
            df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
        else:
            df = pd.read_sql(f"SELECT * FROM {source_table}", conn_stg)
    except Exception as e:
        print(f"❌ Erreur lecture source {source_table} : {e}")
        return

    # 2. Appliquer les jointures
    for fk_col, dim_ref in join_keys.items():
        dim_table, dim_col = dim_ref.split(".")

        if fk_col == "FK_Temps":
            # ⚠ Si pas de colonne date dans le staging → on skip la jointure
            if "Date_Match" in df.columns:
                df = df.rename(columns={"Date_Match": "Join_Date"})
            elif "Date_Entrainement" in df.columns:
                df = df.rename(columns={"Date_Entrainement": "Join_Date"})
            elif "Date_Examen" in df.columns:
                df = df.rename(columns={"Date_Examen": "Join_Date"})
            else:
                print(f"⚠️ Pas de colonne de date détectée pour 'FK_Temps' dans '{table_name}'")
                continue

            time_df = pd.read_sql("SELECT ID_Temps AS FK_Temps, Date AS Join_Date FROM dim_time", conn_dwh)
            df["Join_Date"] = pd.to_datetime(df["Join_Date"]).dt.date
            time_df["Join_Date"] = pd.to_datetime(time_df["Join_Date"]).dt.date
            # ⚠ Garde uniquement les dates dans dim_time
            df = df[df["Join_Date"].isin(time_df["Join_Date"])]
            
            df = df.merge(time_df, on="Join_Date", how="left").drop(columns=["Join_Date"])

        else:
            col_guess = fk_col.replace("FK_", "ID_")
            if col_guess in df.columns:
                df = df.rename(columns={col_guess: fk_col})

    # 3. Forcer les FK float → int
    for col in df.columns:
        if col.startswith("FK_") and df[col].dtype == "float64":
            df[col] = df[col].apply(lambda x: int(x) if pd.notna(x) else None)

    # 4. Générer clé primaire technique
    if auto_pk and pk_name not in df.columns:
        df.insert(0, pk_name, range(1, len(df) + 1))


    df = df[[col for col in ([pk_name] if auto_pk else []) + columns if col in df.columns]]

        # 5. Créer la table si elle n'existe pas (avec clé primaire composée)
    dtype_map = {"int64": "INT", "float64": "FLOAT", "object": "VARCHAR(255)", "bool": "BIT"}
    create_cols = []
    pk_fields = [pk_name] if auto_pk else []
    pk_fields += [col for col in df.columns if col.startswith("FK_")]

    for col in df.columns:
        dtype = dtype_map.get(str(df[col].dtype), "VARCHAR(255)")
        line = f"{col} {dtype}"
        create_cols.append(line)

    pk_constraint = f", PRIMARY KEY ({', '.join(pk_fields)})" if pk_fields else ""

    create_sql = f"""
    IF NOT EXISTS (SELECT * FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_NAME = '{table_name}')
    BEGIN
        CREATE TABLE {table_name} (
            {', '.join(create_cols)}
            {pk_constraint}
        )
    END
    """
    cursor_dwh.execute(create_sql)
    conn_dwh.commit()


    # 6. 🔁 UPSERT ligne par ligne (FKs comme clé de recherche)
    print(f"🚀 Synchronisation de '{table_name}' ({len(df)} lignes)...")
    upsert_keys = [col for col in df.columns if col.startswith("FK_")]

    for _, row in tqdm(df.iterrows(), total=len(df)):
        row_clean = [clean_sql_value(v) for v in row]
        row_dict = dict(zip(df.columns, row_clean))

        where_clause = " AND ".join([f"{key} = ?" for key in upsert_keys])
        check_sql = f"SELECT COUNT(*) FROM {table_name} WHERE {where_clause}"
        check_values = [row_dict[k] for k in upsert_keys]

        cursor_dwh.execute(check_sql, *check_values)
        exists = cursor_dwh.fetchone()[0]

        if exists:
            update_cols = [c for c in df.columns if c not in upsert_keys]
            update_sql = f"UPDATE {table_name} SET " + ", ".join([f"{col} = ?" for col in update_cols]) + f" WHERE {where_clause}"
            update_vals = [row_dict[c] for c in update_cols] + check_values
            cursor_dwh.execute(update_sql, *update_vals)
        else:
            insert_sql = f"INSERT INTO {table_name} ({', '.join(df.columns)}) VALUES ({', '.join(['?'] * len(df.columns))})"
            cursor_dwh.execute(insert_sql, *row_clean)

    conn_dwh.commit()
    print(f"✅ Table '{table_name}' synchronisée avec succès.")

In [6]:
success = []
errors = []

with open("dwh_config.json", encoding="utf-8") as f:
    configs = json.load(f)

for config in configs:
    if config["type"] == "dimension":
        upsert_dimension_from_config(config, conn_stg, conn_dwh)

    elif config["type"] == "fact":
        table_name = config["table"]

        # 🔒 Skip si la source est une liste (non supporté)
        if not isinstance(config["source"], str):
            print(f"⚠️ Table '{table_name}' ignorée (source multiple non supportée)")
            continue

        try:
            print(f"\n🚀 Chargement de la table de faits : {table_name}")
            upsert_fact_from_config(config, conn_stg, conn_dwh)
            success.append(table_name)
        except Exception as e:
            print(f"❌ Erreur dans '{table_name}': {str(e)}")
            errors.append({"table": table_name, "error": str(e)})

# ✅ Résumé final
if success:
    print("\n✅ Tables chargées avec succès :")
    for t in success:
        print(f"   - {t}")

if errors:
    print("\n❌ Erreurs détectées :")
    for err in errors:
        print(f"   🛑 {err['table']}: {err['error']}")


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\3304619568.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT 

✅ Dimension 'dim_joueur' synchronisée avec mise à jour intelligente.
✅ Dimension 'dim_match' synchronisée avec mise à jour intelligente.
✅ Dimension statique 'dim_time' existe déjà. Ignorée.
✅ Dimension statique 'dim_objectif' existe déjà. Ignorée.


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\3304619568.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)


✅ Dimension 'dim_entrainement' synchronisée avec mise à jour intelligente.
✅ Dimension statique 'dim_resultat' existe déjà. Ignorée.
✅ Dimension statique 'dim_position' existe déjà. Ignorée.
✅ Dimension statique 'dim_pied' existe déjà. Ignorée.
✅ Dimension statique 'dim_competition' existe déjà. Ignorée.
✅ Dimension statique 'dim_categorie' existe déjà. Ignorée.
✅ Dimension statique 'dim_meteo' existe déjà. Ignorée.
✅ Dimension statique 'dim_etat_terrain' existe déjà. Ignorée.

🚀 Chargement de la table de faits : faits_matchs


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  time_df = pd.read_sql("SELECT I

...

❌ Erreur dans 'faits_examens_locomoteurs': ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot insert the value NULL into column 'FK_Temps', table 'DWH_ClubAfricain.dbo.faits_examens_locomoteurs'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")

🚀 Chargement de la table de faits : faits_examens_orl
🚀 Synchronisation de 'faits_examens_orl' (744 lignes)...


  6%|████▎                                                                          | 41/744 [00:00<00:00, 1584.23it/s]
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_10448\2851229998.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBA

❌ Erreur dans 'faits_examens_orl': ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot insert the value NULL into column 'FK_Temps', table 'DWH_ClubAfricain.dbo.faits_examens_orl'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")

🚀 Chargement de la table de faits : faits_examens_stomatologiques
🚀 Synchronisation de 'faits_examens_stomatologiques' (760 lignes)...


  4%|███▎                                                                           | 32/760 [00:00<00:00, 2000.26it/s]

❌ Erreur dans 'faits_examens_stomatologiques': ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot insert the value NULL into column 'FK_Temps', table 'DWH_ClubAfricain.dbo.faits_examens_stomatologiques'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")

✅ Tables chargées avec succès :
   - faits_matchs
   - faits_performances
   - faits_entrainements
   - faits_objectifs
   - faits_psychologiques

❌ Erreurs détectées :
   🛑 faits_examens_generaux: ('23000', "[23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]Cannot insert the value NULL into column 'FK_Temps', table 'DWH_ClubAfricain.dbo.faits_examens_generaux'; column does not allow nulls. INSERT fails. (515) (SQLExecDirectW); [23000] [Microsoft][ODBC Driver 17 for SQL Server][SQL Server]The statement has been terminated. (3621)")
   🛑 faits_examens_cardiopulmonaires: ('230

In [ ]:
import time
from datetime import datetime

def log_execution(status, success_tables, error_tables):
    log_path = "dwh_log.txt"
    with open(log_path, "a", encoding="utf-8") as f:
        f.write(f"\n=== Exécution à {datetime.now().strftime('%Y-%m-%d %H:%M:%S')} ===\n")
        f.write(f"Statut : {'✅ Succès' if status else '❌ Échecs'}\n")
        if success_tables:
            f.write("Tables chargées avec succès :\n")
            for t in success_tables:
                f.write(f"  - {t}\n")
        if error_tables:
            f.write("Erreurs rencontrées :\n")
            for err in error_tables:
                f.write(f"  🛑 {err['table']}: {err['error']}\n")
        f.write("=" * 50 + "\n")

# Boucle symbolique toutes les 30 minutes (stop avec Kernel > Interrupt)
while True:
    success = []
    errors = []

    with open("dwh_config.json", encoding="utf-8") as f:
        configs = json.load(f)

    for config in configs:
        if config["type"] == "dimension":
            upsert_dimension_from_config(config, conn_stg, conn_dwh)
        elif config["type"] == "fact":
            table_name = config["table"]
            if not isinstance(config["source"], str):
                print(f"⚠️ Table '{table_name}' ignorée (source multiple non supportée)")
                continue
            try:
                print(f"\n🚀 Chargement de la table de faits : {table_name}")
                upsert_fact_from_config(config, conn_stg, conn_dwh)
                success.append(table_name)
            except Exception as e:
                print(f"❌ Erreur dans '{table_name}': {str(e)}")
                errors.append({"table": table_name, "error": str(e)})

    log_execution(status=(len(errors) == 0), success_tables=success, error_tables=errors)
    
    print("\n⏳ Attente de 30 minutes avant la prochaine exécution...")
    time.sleep(30 * 60)  # ⏱ 30 minutes


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\3304619568.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT 

✅ Dimension 'dim_joueur' synchronisée avec mise à jour intelligente.
✅ Dimension 'dim_match' synchronisée avec mise à jour intelligente.
✅ Dimension statique 'dim_time' existe déjà. Ignorée.
✅ Dimension statique 'dim_objectif' existe déjà. Ignorée.


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\3304619568.py:23: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\3304619568.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)


✅ Dimension 'dim_entrainement' synchronisée avec mise à jour intelligente.
✅ Dimension statique 'dim_resultat' existe déjà. Ignorée.
✅ Dimension statique 'dim_position' existe déjà. Ignorée.
✅ Dimension statique 'dim_pied' existe déjà. Ignorée.
✅ Dimension statique 'dim_competition' existe déjà. Ignorée.
✅ Dimension statique 'dim_categorie' existe déjà. Ignorée.
✅ Dimension statique 'dim_meteo' existe déjà. Ignorée.
✅ Dimension statique 'dim_etat_terrain' existe déjà. Ignorée.

🚀 Chargement de la table de faits : faits_matchs


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:40: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  time_df = pd.read_sql("SELECT I

...

100%|██████████████████████████████████████████████████████████████████████████| 102672/102672 [33:00<00:00, 51.85it/s]
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table}", conn_stg)


✅ Table 'faits_entrainements' synchronisée avec succès.

🚀 Chargement de la table de faits : faits_objectifs
🚀 Synchronisation de 'faits_objectifs' (1458 lignes)...


100%|████████████████████████████████████████████████████████████████████████████| 1458/1458 [00:01<00:00, 1001.09it/s]
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_24352\1612221268.py:17: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table} WHERE Actif = 1", conn_stg)


✅ Table 'faits_objectifs' synchronisée avec succès.

🚀 Chargement de la table de faits : faits_psychologiques
🚀 Synchronisation de 'faits_psychologiques' (21027 lignes)...


 81%|█████████████████████████████████████████████████████████████              | 17137/21027 [01:09<00:18, 211.49it/s]

In [12]:
import json

# Charger la configuration
with open("dwh_config.json") as f:
    full_config = json.load(f)

# Extraire la config de la table 'faits_objectifs'
config_faits_objectifs = next(t for t in full_config if t["table"] == "faits_objectifs")

# Changer temporairement la source vers ta vue
config_faits_objectifs["source"] = "vw_faits_objectifs"

# Appel de la fonction de chargement
upsert_fact_from_config(config_faits_objectifs, conn_stg, conn_dwh)


C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_22012\1612221268.py:15: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  test_df = pd.read_sql(f"SELECT TOP 0 * FROM {source_table}", conn_stg)
C:\Users\MOEµNESS\AppData\Local\Temp\ipykernel_22012\1612221268.py:19: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(f"SELECT * FROM {source_table}", conn_stg)


🚀 Synchronisation de 'faits_objectifs' (1458 lignes)...


100%|████████████████████████████████████████████████████████████████████████████| 1458/1458 [00:00<00:00, 1583.48it/s]

✅ Table 'faits_objectifs' synchronisée avec succès.
